# MLD cross-validation

This notebook tunes the pruned maximum-likelihood decoder used by `decoder.py` on a deterministic subset of the labelled training data. Each candidate is evaluated separately for the `X` and `Z` bases using cross-validation, and the selected configuration minimizes the combined logical error rate (LER). Runtime is reported as a secondary metric.

The folds are used only for evaluation: MLD has no trainable parameters and is not fitted on the fold data.


In [36]:
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path("..").resolve()
REPO_ROOT = REPO_ROOT.resolve()
requirements_path = REPO_ROOT / "submission_template" / "requirements.txt"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(requirements_path)])
print("installed dependencies for:", sys.executable)


installed dependencies for: /Users/suneg/Dev/QIS-hackathon/qis-my-error/.venv/bin/python



[notice] A new release of pip available: 22.3.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [37]:
import time
from itertools import product

import numpy as np
import stim

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "submission_template"))

from hackathon_2026 import load_kaggle_csv, logical_error_rate, problem_info

# Reload MLD from this repository so a previously imported/stale module cannot hide
# the log-weight API used by decoder.py.
import importlib
sys.modules.pop("mld", None)
mld = importlib.import_module("mld")
if not all(hasattr(mld, name) for name in ("build_mld", "marginals_logw", "mld_logw")):
    raise ImportError(f"loaded unexpected MLD module: {getattr(mld, '__file__', None)}")
build_mld = mld.build_mld
marginals_logw = mld.marginals_logw
mld_logw = mld.mld_logw

DATA_PATH = REPO_ROOT / "data" / "train" / "train.csv"
SUBSET_PER_BASIS = 7500
N_FOLDS = 3
SEED = 42
print("repository:", REPO_ROOT)
print("data:", DATA_PATH)


repository: /Users/suneg/Dev/QIS-hackathon/qis-my-error
data: /Users/suneg/Dev/QIS-hackathon/qis-my-error/data/train/train.csv


## Load labelled data and MLD inputs

For each basis, the notebook loads detector events and logical labels, then extracts `H`, `L`, and `p` from the corresponding Stim circuit. A separate MLD table is built for each basis and each tested `hexp` value.


In [38]:
def load_basis_data(data_path: str | Path, subset_per_basis: int, seed: int):
    data = load_kaggle_csv(str(data_path))
    if "observable_flips" not in data:
        raise ValueError("cross-validation requires L0-L3 labels")
    result = {}
    for offset, basis in enumerate(("X", "Z")):
        rows = np.flatnonzero(data["basis"] == basis)
        if len(rows) == 0:
            raise ValueError(f"no {basis} rows found")
        rng = np.random.default_rng(seed + offset)
        rng.shuffle(rows)
        if subset_per_basis is not None:
            rows = rows[:min(subset_per_basis, len(rows))]
        result[basis] = {
            "events": data["detector_events"][rows].astype(np.uint8),
            "labels": data["observable_flips"][rows].astype(bool),
        }
    return result

def load_problem_data(data_folder: str | Path):
    problems = {}
    for basis in ("X", "Z"):
        circuit = stim.Circuit.from_file(str(Path(data_folder) / f"{basis}_circuit.stim"))
        info = problem_info(circuit)
        problems[basis] = info
        print(f"{basis}: H={info['H'].shape}, L={info['L'].shape}, p={info['p'].shape}")
    return problems

basis_data = load_basis_data(DATA_PATH, SUBSET_PER_BASIS, SEED)
problems = load_problem_data(DATA_PATH.parent)
for basis in ("X", "Z"):
    print(basis, basis_data[basis]["events"].shape, basis_data[basis]["labels"].shape)


X: H=(70, 2470), L=(4, 2470), p=(2470,)
Z: H=(70, 2465), L=(4, 2465), p=(2465,)
X (7500, 70) (7500, 4)
Z (7500, 70) (7500, 4)


## Deterministic cross-validation folds

In [39]:
def make_folds(size: int, n_folds: int, seed: int) -> list[np.ndarray]:
    if n_folds < 2 or n_folds > size:
        raise ValueError("n_folds must be at least 2 and no larger than the subset size")
    indices = np.arange(size)
    np.random.default_rng(seed).shuffle(indices)
    return [fold.copy() for fold in np.array_split(indices, n_folds)]

folds = {
    basis: make_folds(len(basis_data[basis]["events"]), N_FOLDS, SEED + i)
    for i, basis in enumerate(("X", "Z"))
}
print({basis: [len(fold) for fold in folds[basis]] for basis in ("X", "Z")})


{'X': [2500, 2500, 2500], 'Z': [2500, 2500, 2500]}


## MLD parameter grid

Each tier is `(eps, max_states, threshold)`. The first tier always decodes every shot. A later tier re-decodes shots whose current logical marginal is between `threshold` and `1 - threshold`. The grid also tests the `hexp` heuristic and the lower-level `compact_every` performance parameter.


In [ ]:
# Tiers use the same four-field format as decoder.py:
# (direction, eps, max_states, uncertainty_threshold).
# Direction 0 uses the normal detector order; direction 1 uses the reversed order.
BASE_TIERS = ((0, 6.666666666666667e-5, 1792, 0.0), (1, 3e-5, 3072, 0.1))
BASE_HEXP = 0.60
BASE_COMPACT_EVERY = 32


#best configuration: {'tiers': ((0, 0.0001, 1536, 0.0), (1, 3e-05, 4096, 0.15)), 'hexp': 0.6, 'compact_every': 32}



# Tune the production-equivalent two-pass decoder tightly around the current baseline.
TWO_PASS_GRID = [
    {
        "tiers": (
            (0, first_eps, first_states, 0.0),
            (1, second_eps, second_states, threshold),
        ),
        "hexp": hexp,
        "compact_every": compact_every,
    }
    for first_eps, first_states, second_eps, second_states, threshold, hexp, compact_every in product(
        (3e-4, 1e-4),
        (1024, 1536),
        (1e-5, 3e-5),
        (4096, 8192),
        (0.15, 0.20, 0.25),
        (0.55, 0.60),
        (32,),
    )
]

# Test a small number of nearby three-pass configurations.
THREE_PASS_GRID = []#[
    #{
        #"tiers": (
            #(0, 3e-4, 512, 0.0),
            #(1, 3e-5, 4096, middle_threshold),
            #(0, last_eps, last_states, final_threshold),
        #),
        #"hexp": hexp,
        #"compact_every": compact_every,
    #}
    #for middle_threshold, last_eps, last_states, final_threshold, hexp, compact_every in product(
        #(0.10, 0.15),
        #(1e-6, 3e-6),
        #(4096, 8192),
        #(0.25, 0.35),
        #(0.50, 0.55),
        #(32,),
    #)
#]

PARAM_GRID = []
_seen = set()
for candidate in TWO_PASS_GRID + THREE_PASS_GRID:
    key = repr(candidate)
    if key not in _seen:
        _seen.add(key)
        PARAM_GRID.append(candidate)

print(f"testing {len(PARAM_GRID)} production-equivalent MLD configurations")
print("baseline:", {
    "tiers": BASE_TIERS,
    "hexp": BASE_HEXP,
    "compact_every": BASE_COMPACT_EVERY,
})


testing 96 production-equivalent MLD configurations
baseline: {'tiers': ((0, 0.0001, 1536, 0.0), (1, 3e-05, 4096, 0.15)), 'hexp': 0.55, 'compact_every': 32}


In [41]:
TABLE_CACHE = {}

def get_tables(basis: str, hexp: float):
    """Build the same forward/reverse table pair used by decoder._tables."""
    key = (basis, float(hexp))
    if key not in TABLE_CACHE:
        info = problems[basis]
        TABLE_CACHE[key] = (
            build_mld(info["H"], info["L"], info["p"], hexp=hexp),
            build_mld(info["H"][::-1].copy(), info["L"], info["p"], hexp=hexp),
        )
    return TABLE_CACHE[key]


def decode_production_marginals(tables, detector_events, tiers, compact_every):
    """Match decoder.decode_marginals for arbitrary candidate tier parameters."""
    D = np.asarray(detector_events, dtype=bool)
    events = (np.ascontiguousarray(D), np.ascontiguousarray(D[:, ::-1]))
    logw = None
    for direction, eps, max_states, threshold in tiers:
        if logw is None:
            logw = mld_logw(
                tables[direction],
                events[direction],
                eps=eps,
                max_states=max_states,
                compact_every=compact_every,
            )
            continue
        selected = np.flatnonzero(
            (np.abs(marginals_logw(logw) - 0.5) < 0.5 - threshold).any(1)
        )
        if len(selected):
            logw[selected] = np.maximum(
                logw[selected],
                mld_logw(
                    tables[direction],
                    events[direction][selected],
                    eps=eps,
                    max_states=max_states,
                    compact_every=compact_every,
                ),
            )
    return marginals_logw(logw)


def evaluate_configuration(params: dict) -> dict:
    basis_scores = {"X": [], "Z": []}
    total_wrong = 0
    total_bits = 0
    total_seconds = 0.0
    for basis in ("X", "Z"):
        events = basis_data[basis]["events"]
        labels = basis_data[basis]["labels"]
        tables = get_tables(basis, params["hexp"])
        for fold_index, test_indices in enumerate(folds[basis]):
            started = time.perf_counter()
            marginals = decode_production_marginals(
                tables,
                events[test_indices],
                params["tiers"],
                params["compact_every"],
            )
            predictions = (marginals > 0.5).astype(np.uint8)
            elapsed = time.perf_counter() - started
            score = logical_error_rate(predictions, labels[test_indices])
            wrong = int(np.count_nonzero(predictions.astype(bool) != labels[test_indices]))
            total_wrong += wrong
            total_bits += predictions.size
            total_seconds += elapsed
            basis_scores[basis].append({
                "fold": fold_index,
                "ler": score,
                "seconds": elapsed,
            })
    return {
        "params": params,
        "combined_ler": total_wrong / total_bits,
        "seconds": total_seconds,
        "basis": basis_scores,
    }


## Adaptive coordinate search around the baseline

This search optimizes the two-pass production decoder without requiring a fixed large grid. It first evaluates one step below and above the current value for each parameter. If a neighboring value improves the cross-validated LER, the search moves there and keeps testing one further step in that same direction. A parameter is considered converged when neither neighbor improves the objective. The search repeats coordinate sweeps until no parameter moves.

The objective is combined LER, with runtime breaking ties. The safety limits prevent an unbounded search; increase them only after checking that the search remains physically meaningful.


In [42]:
# Coordinate-search settings. This optimizes the two-pass baseline only.
# The production sweep directions remain fixed at forward (0) then reverse (1).
COORDINATE_MAX_ROUNDS = 30
COORDINATE_MAX_STEPS_PER_PARAMETER = 6

PARAMETER_NAMES = (
    "first_eps",
    "first_states",
    "second_eps",
    "second_states",
    "threshold",
    "hexp",
    "compact_every",
)


def point_from_baseline():
    first, second = BASE_TIERS
    return {
        "first_eps": first[1],
        "first_states": first[2],
        "second_eps": second[1],
        "second_states": second[2],
        "threshold": second[3],
        "hexp": BASE_HEXP,
        "compact_every": BASE_COMPACT_EVERY,
    }


def params_from_point(point):
    return {
        "tiers": (
            (0, point["first_eps"], int(point["first_states"]), 0.0),
            (1, point["second_eps"], int(point["second_states"]), point["threshold"]),
        ),
        "hexp": point["hexp"],
        "compact_every": int(point["compact_every"]),
    }


# A step is chosen in the natural scale of each parameter. Eps uses a factor
# because its useful values span orders of magnitude; state counts use doubling.
def neighboring_points(point, name):
    lower = point.copy()
    upper = point.copy()
    value = point[name]
    if name in ("first_eps", "second_eps"):
        lower[name] = value / 1.5
        upper[name] = value * 1.5
        lower[name] = max(lower[name], 1e-7)
        upper[name] = min(upper[name], 1e-2)
    elif name in ("first_states", "second_states"):
        lower[name] = max(128, value - 256)
        upper[name] = min(32768, value + 256)
    elif name in ("threshold", "hexp"):
        lower[name] = max(0.0, value - 0.025)
        upper[name] = min(0.95 if name == "threshold" else 1.5, value + 0.025)
    elif name == "compact_every":
        lower[name] = max(4, value - 8)
        upper[name] = min(128, value + 8)
    else:
        raise ValueError(f"unknown MLD parameter: {name}")
    return lower, upper


def objective_key(result):
    return (result["combined_ler"], result["seconds"])


coordinate_results = {}
def evaluate_point(point):
    key = tuple((name, point[name]) for name in PARAMETER_NAMES)
    if key not in coordinate_results:
        params = params_from_point(point)
        print("evaluating:", params)
        coordinate_results[key] = evaluate_configuration(params)
    return coordinate_results[key]


def adaptive_coordinate_search():
    point = point_from_baseline()
    current = evaluate_point(point)
    movement_history = []

    for round_index in range(1, COORDINATE_MAX_ROUNDS + 1):
        moved_this_round = False
        print(f"\ncoordinate-search round {round_index}")
        for name in PARAMETER_NAMES:
            steps = 0
            while steps < COORDINATE_MAX_STEPS_PER_PARAMETER:
                lower, upper = neighboring_points(point, name)
                candidates = [(point, current, "current")]
                if lower[name] != point[name]:
                    candidates.append((lower, evaluate_point(lower), "lower"))
                if upper[name] != point[name]:
                    candidates.append((upper, evaluate_point(upper), "higher"))
                best_point, best_result, direction = min(
                    candidates,
                    key=lambda item: objective_key(item[1]),
                )
                if direction == "current":
                    break
                old_value = point[name]
                point = best_point
                current = best_result
                movement_history.append({
                    "round": round_index,
                    "parameter": name,
                    "direction": direction,
                    "old_value": old_value,
                    "new_value": point[name],
                    "combined_ler": current["combined_ler"],
                })
                moved_this_round = True
                steps += 1
                print(
                    f"moved {name} {direction}: {old_value} -> {point[name]} "
                    f"(LER={current['combined_ler']:.6f})"
                )
        if not moved_this_round:
            print("converged: no parameter moved in this round")
            break
    else:
        print("stopped at COORDINATE_MAX_ROUNDS safety limit")

    return point, current, movement_history


coordinate_best_point, coordinate_best, coordinate_history = adaptive_coordinate_search()
print("\ncoordinate-search result:", params_from_point(coordinate_best_point))
print(f"coordinate-search LER: {coordinate_best['combined_ler']:.6f}")
print(f"coordinate-search runtime: {coordinate_best['seconds']:.2f}s")
print("movement history:")
for movement in coordinate_history:
    print(movement)


evaluating: {'tiers': ((0, 0.0001, 1536, 0.0), (1, 3e-05, 4096, 0.15)), 'hexp': 0.55, 'compact_every': 32}

coordinate-search round 1
evaluating: {'tiers': ((0, 6.666666666666667e-05, 1536, 0.0), (1, 3e-05, 4096, 0.15)), 'hexp': 0.55, 'compact_every': 32}
evaluating: {'tiers': ((0, 0.00015000000000000001, 1536, 0.0), (1, 3e-05, 4096, 0.15)), 'hexp': 0.55, 'compact_every': 32}
moved first_eps lower: 0.0001 -> 6.666666666666667e-05 (LER=0.034633)
evaluating: {'tiers': ((0, 4.4444444444444447e-05, 1536, 0.0), (1, 3e-05, 4096, 0.15)), 'hexp': 0.55, 'compact_every': 32}
moved first_eps lower: 6.666666666666667e-05 -> 4.4444444444444447e-05 (LER=0.034567)
evaluating: {'tiers': ((0, 2.962962962962963e-05, 1536, 0.0), (1, 3e-05, 4096, 0.15)), 'hexp': 0.55, 'compact_every': 32}
evaluating: {'tiers': ((0, 4.4444444444444447e-05, 1280, 0.0), (1, 3e-05, 4096, 0.15)), 'hexp': 0.55, 'compact_every': 32}
evaluating: {'tiers': ((0, 4.4444444444444447e-05, 1792, 0.0), (1, 3e-05, 4096, 0.15)), 'hexp': 0

## Run the search

The objective is the combined LER across all held-out `X` and `Z` folds. Ties are broken by total runtime. The first run may spend extra time compiling the Numba kernel.


In [43]:
results = []
print(f"evaluating {len(PARAM_GRID)} MLD configurations")
for index, params in enumerate(PARAM_GRID, start=1):
    result = evaluate_configuration(params)
    results.append(result)
    print(
        f"{index}/{len(PARAM_GRID)} "
        f"LER={result['combined_ler']:.6f} "
        f"time={result['seconds']:.2f}s "
        f"params={params}"
    )

results.sort(key=lambda result: (result["combined_ler"], result["seconds"]))
best = results[0]
print("\nbest configuration:", best["params"])
print(f"best combined LER: {best['combined_ler']:.6f}")
print(f"best CV runtime: {best['seconds']:.2f}s")


evaluating 96 MLD configurations
1/96 LER=0.034717 time=7.48s params={'tiers': ((0, 0.0003, 1024, 0.0), (1, 1e-05, 4096, 0.15)), 'hexp': 0.55, 'compact_every': 32}
2/96 LER=0.034800 time=4.88s params={'tiers': ((0, 0.0003, 1024, 0.0), (1, 1e-05, 4096, 0.15)), 'hexp': 0.6, 'compact_every': 32}
3/96 LER=0.034800 time=5.79s params={'tiers': ((0, 0.0003, 1024, 0.0), (1, 1e-05, 4096, 0.2)), 'hexp': 0.55, 'compact_every': 32}
4/96 LER=0.034850 time=4.88s params={'tiers': ((0, 0.0003, 1024, 0.0), (1, 1e-05, 4096, 0.2)), 'hexp': 0.6, 'compact_every': 32}
5/96 LER=0.034883 time=5.10s params={'tiers': ((0, 0.0003, 1024, 0.0), (1, 1e-05, 4096, 0.25)), 'hexp': 0.55, 'compact_every': 32}
6/96 LER=0.034967 time=4.55s params={'tiers': ((0, 0.0003, 1024, 0.0), (1, 1e-05, 4096, 0.25)), 'hexp': 0.6, 'compact_every': 32}
7/96 LER=0.034700 time=5.46s params={'tiers': ((0, 0.0003, 1024, 0.0), (1, 1e-05, 8192, 0.15)), 'hexp': 0.55, 'compact_every': 32}
8/96 LER=0.034800 time=5.04s params={'tiers': ((0, 0.00

KeyboardInterrupt: 

In [ ]:
for basis in ("X", "Z"):
    print(f"\n{basis} folds")
    for fold_result in best["basis"][basis]:
        print(fold_result)

print("\nUse these values in decoder.py:")
print("MLD_TIERS =", best["params"]["tiers"])
print("MLD_COMPACT_EVERY =", best["params"]["compact_every"])
print("MLD_HEXP =", best["params"]["hexp"])
print("\nRerun the selected configuration on a fresh holdout before trusting the result.")
